In [ ]:
import langchain, pandas, numpy, seaborn, matplotlib
import warnings
import langchain_community
from langchain_community.document_loaders import PyMuPDFLoader,
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownTextSplitter
import pandas as pd, numpy as np
import tiktoken
from langchain_classic import vectorstores, FAISS
from langchain_chroma import Chroma
from langchain_openai import *
import os, dotenv
from dotenv import load_dotenv
load_dotenv()

warnings.filterwarnings('ignore')

In [6]:
pdf_path =r"C:\Users\Neel\Desktop\Azure_open_ai\Loader\pdfs\pandas_datetime_book.pdf"

In [7]:
pymu_loader = PyMuPDFLoader(file_path=pdf_path)
pdf_docs = list(pymu_loader.lazy_load())

meta_data=[]
page_content=[]
for doc in pdf_docs:
  page_content.append(doc.page_content)
  meta_data.append(doc.metadata)
df = pd.DataFrame({
  "meta_data": meta_data,
  "page_content" : page_content
})


enc=tiktoken.encoding_for_model('gpt-3.5-turbo')

df['word_count']=df['page_content'].apply(lambda x: len(x.split()))
df['sent_count']=df['page_content'].apply(lambda x: len(x.split(".")))
df['char_count']=df['page_content'].apply(lambda x: len(x))
df['token_count']=np.round(df['word_count'] * 1.3,0).astype('int')
df['tik_token']= df['page_content'].apply(lambda x: len(enc.encode(x)))


  


df.describe()
  

,word_count,sent_count,char_count,token_count,tik_token
count,43.000000,43.000000,43.000000,43.000000,43.000000
mean,175.767442,182.720930,1658.697674,228.488372,514.302326
std,65.288677,749.253544,926.039198,84.950230,236.940774
min,11.000000,1.000000,77.000000,14.000000,29.000000
25%,156.000000,9.000000,1210.000000,203.000000,377.000000
50%,191.000000,22.000000,1759.000000,248.000000,566.000000
75%,219.000000,28.500000,2020.500000,285.000000,687.000000
max,309.000000,3949.000000,5178.000000,402.000000,925.000000


In [8]:
spliter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=60)
chunks= spliter.split_documents(pdf_docs)

meta_data=[]
page_content=[]


for doc in chunks:
  page_content.append(doc.page_content)
  meta_data.append(doc.metadata)
dfs = pd.DataFrame({
  "meta_data": meta_data,
  "page_content" : page_content
})


enc=tiktoken.encoding_for_model('gpt-3.5-turbo')

dfs['word_count']=df['page_content'].apply(lambda x: len(x.split()))
dfs['sent_count']=df['page_content'].apply(lambda x: len(x.split(".")))
dfs['char_count']=df['page_content'].apply(lambda x: len(x))
dfs['token_count']=np.round(df['word_count'] * 1.3,0).astype('int')
dfs['tik_token']= df['page_content'].apply(lambda x: len(enc.encode(x)))
dfs.describe()


,word_count,sent_count,char_count,token_count,tik_token
count,43.000000,43.000000,43.000000,43.000000,43.000000
mean,175.767442,182.720930,1658.697674,228.488372,514.302326
std,65.288677,749.253544,926.039198,84.950230,236.940774
min,11.000000,1.000000,77.000000,14.000000,29.000000
25%,156.000000,9.000000,1210.000000,203.000000,377.000000
50%,191.000000,22.000000,1759.000000,248.000000,566.000000
75%,219.000000,28.500000,2020.500000,285.000000,687.000000
max,309.000000,3949.000000,5178.000000,402.000000,925.000000


In [25]:
api_key = os.getenv("OPENAI_API_KEY")
embed_model=OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512, api_key=api_key)

if os.path.exists("my_dir_1"):
  print("VS is there no need to recreate")
  # load
  
  chorma_db=Chroma(
        embedding_function=embed_model,
    collection_name='new_collection',
    persist_directory='my_dir_1',
   
  )



else:  


  chorma_db = Chroma.from_documents(
    embedding=embed_model,
    collection_name='new_collection',
    persist_directory='my_dir_1',
     documents=chunks
  )

  

VS is there no need to recreate


In [40]:
retriver=chorma_db.as_retriever(search_type='mmr', 
                       search_kwargs={'k':2, 'lambda_mult':1})

In [41]:
q="What is the date time function in pandas ?"

In [42]:
rr=retriver.invoke(q)
type(rr[1])

langchain_core.documents.base.Document

In [43]:
print(rr[0].page_content)
print(rr[1].page_content)

down to individual seconds. 
 
1.1  Why Date Handling Matters 
Raw data often stores dates as plain strings such as '2024-07-15' or integers like 20240715. Before you 
can slice, aggregate, or compute differences, these must be converted to proper datetime objects. 
Pandas makes this conversion seamless and then exposes hundreds of vectorised operations through 
a single .dt accessor. 
 
1.2  Core Date/Time Types in Pandas 
pandas.Timestamp — Represents a single point in time. It is pandas' equivalent of Python's 
datetime.datetime and is the fundamental building block.
Kuntal Chakraborty 
Page 4  |  Cracking Pandas DateTime — Kuntal Chakraborty 
Chapter 1 — Introduction to Dates & Times in Pandas 
Date and time handling is one of the most critical skills in data science and analytics. Whether you are 
analysing sales trends, monitoring server logs, forecasting stock prices, or studying patient records, 
almost every real-world dataset contains a time dimension. Python's pandas library